# Knowledge Based Engineering: Group 12 Notebook

## Step 1 - Identify the task

- Develop self-driving trailer agent as a knowldege-based agentto make safe driving decision with minimal risk.
- Helping the self-driving trailer get from (Nairobi  → Kisumu)
- Actions to do (stop, slow down, go) in the course of the route to the destination.



## Step 2: Assemble relevant Knowledge

Things to know:
- Towns and roads between them(distances in km)
- Traffic Light color
- Action to take based on colour
- Traffic laws and road conditions (current weather conditions)
- Blocked roads

API declaration:
1. Open-Meteo weather API - It is used to get the current rainfall at a town(free with no key needed)
https://api.open-meteo.com/v1/forecast
Terms: https://open-meteo.com/en/terms

2. OpenStreetMap Nominatim: gets the latitude/longitude of each town

   https://nominatim.openstreetmap.org/search
   
   Usage policy: https://operations.osmfoundation.org/policies/nominatim/


In [13]:
# Imports for all the cells
import requests # lets Python ask a website / API for data
import time # Lets us pause btn requests

In [14]:
# Nominatim's policy
HEADERS = {"User-Agent": "Knowledge_Based_Agent_Trailer/1.0 (student project; amani.koros@strathmore.edu)"}

TOWNS = ["Nairobi", "Naivasha", "Narok", "Nakuru", "Bomet", "Litein",
         "Kisii", "Eldoret", "Kericho", "HomaBay", "Kisumu"]
SEARCH_NAME = {"HomaBay": "Homa Bay"}          # OpenStreetMap spells this one with a space

def get_coordinates(town):
    # Ask OpenStreetMap (Nominatim) for the (latitude, longitude) of a town in Kenya.
    params = {"q": f"{SEARCH_NAME.get(town, town)}, Kenya", "format": "json", "limit": 1,
              "countrycodes": "ke",             # only look inside Kenya
              "featuretype": "settlement"}      # only towns/cities/villages, not counties or wards
    try:
        r = requests.get("https://nominatim.openstreetmap.org/search",
                         params=params, headers=HEADERS, timeout=10)
        first = r.json()[0]                                              # best match
        return float(first["lat"]), float(first["lon"]), first["type"]   # lat/lon arrive as text, so convert
    except Exception:
        return None                                                      # no internet or town not found

# Look up every town once and keep the answers in dictionaries
TOWN_COORDS = {}
TOWN_TYPE = {}
for town in TOWNS:
    result = get_coordinates(town)
    if result:
        lat, lon, kind = result
        TOWN_COORDS[town] = (lat, lon)
        TOWN_TYPE[town] = kind                                # what OpenStreetMap says it found (town, city, village...)
    time.sleep(1)                                             # respect the 1 request per second rule

# Town-centre coordinates checked by hand on openstreetmap.org (Narok, Bomet, Homa Bay, were not dipslaying the correct values using the openstreeetmap api, hence we have inputted the values manually)
VERIFIED = {"Narok": (-1.08447, 35.87173),
            "Bomet": (-0.77843, 35.34370),
            "HomaBay": (-0.5684, 34.4758)}
TOWN_COORDS.update(VERIFIED)

print("Coordinates from OpenStreetMap:")
for town, (lat, lon) in TOWN_COORDS.items():
    print(f"  {town:9} {lat:.4f}, {lon:.4f}  ({TOWN_TYPE[town]})")

def get_weather(town):
    # Ask Open-Meteo how much rain (mm) is falling at a town right now, and turn it into a simple label.
    if town not in TOWN_COORDS:
        return None                                           # if they are no coordinates for this town
    lat, lon = TOWN_COORDS[town]
    url = "https://api.open-meteo.com/v1/forecast"
    params = {"latitude": lat, "longitude": lon, "current": "precipitation"}
    try:
        data = requests.get(url, params=params, timeout=10).json()   # send the request, read the answer
        rain_mm = data["current"]["precipitation"]                   # pick out the rainfall measurement in mm
    except Exception:
        return None                                                  # no internet or the API is down
    if rain_mm >= 5:
        return "HeavyRain"
    if rain_mm > 0:
        return "Rain"
    return "Clear"

print("Weather in Nakuru right now:", get_weather("Nakuru"))

Coordinates from OpenStreetMap:
  Nairobi   -1.3024, 36.8289  (historic:administrative)
  Naivasha  -0.7224, 36.4388  (town)
  Narok     -1.0845, 35.8717  (administrative)
  Nakuru    -0.2803, 36.0712  (city)
  Bomet     -0.7784, 35.3437  (administrative)
  Litein    -0.5826, 35.1881  (town)
  Kisii     -0.7389, 34.7540  (administrative)
  Eldoret   0.5198, 35.2715  (city)
  Kericho   -0.3210, 35.2261  (administrative)
  HomaBay   -0.5684, 34.4758  (administrative)
  Kisumu    -0.1029, 34.7542  (city)
Weather in Nakuru right now: Rain


## Step 3: Defined Vocabulary

1.  Predicates: Keys and rules represented in Python
- Road(a,b)km,
- Blocked(a,b),
- Light(colour),
- RoadType(a,b),
- SpeedLimit(type),
- Condition(town,c)

2. Constants: All towns in the routes.
- { Nairobi → Kisumu};
- Lights {Red, Orange, Green};
- Conditions(Clear, Rain, HeavyRain)

## Step 4: Encoding Knowledge about the Domain

- Predicates and Constants defined in Step 3, are now written as Python Dictionaries.

- Each Python dictionary is 1 kind of knowldege

In [15]:
# routes from the directed graph from UCS_trailer
roads = {
    "Nairobi": {"Naivasha": 91, "Narok": 143},
    "Naivasha": {"Nakuru": 71},
    "Narok": {"Bomet": 81},
    "Nakuru": {"Eldoret": 156, "Kericho": 107},
    "Bomet": {"Litein": 44.1,"Kisii": 82.4},
    "Eldoret":{"Kisumu": 138},
    "Kericho":{"Kisumu": 84},
    "Litein":{"Kericho": 32},
    "Kisii":{"HomaBay": 52},
    "HomaBay":{"Kisumu": 108},
    "Kisumu":{},
}

# Light(colour)
light_rules = {
    "Red": "Stop, it is unsafe to move.",
    "Yellow": "Slow down and prepare to stop.",
    "Green": "Go: it is safe to proceed."
}

# Condition(c) -> weather / road condition
condition_rules = {
    "Clear": "Drive safely.",
    "Rain": "Slow down and keep a longer distance between the car infront.",
    "HeavyRain": "Unsafe: stop and wait"
}

# Traffic law -- Speed limit of 80 km/h
speed_limits = {"Highway": 80}
road_types = {("Nairobi", "Naivasha"): "Highway",
              ("Naivasha", "Nakuru"): "Highway"}

# Blocked(a,b) -> Empty at first then continued in Step 5
blocked_roads = set()

# Quick test: can the Knowledge based agent print some output:
print(light_rules["Green"])
print("Speed limit Nairobi-> Naivasha:", speed_limits[road_types[("Nairobi", "Naivasha")]], "km/h")

Go: it is safe to proceed.
Speed limit Nairobi-> Naivasha: 80 km/h


## Step 5: Add New Facts to Knowledge Base Agent

In [16]:
def show_kb():
    # Print everything the agent currently knows (apart from the road distances, which are in the graph).
    print("\n--- Current Knowledge Base ---")
    print("Light rules:")
    for light, action in light_rules.items():
        print(f"  {light:10} ➜ {action}")
    print("Condition rules:")
    for cond, action in condition_rules.items():
        print(f"  {cond:10} ➜ {action}")
    print("Blocked roads:", sorted(blocked_roads) if blocked_roads else "none")
    print("-------------------------------")

def add_new_fact(color, action):
    # Add a new traffic light fact directly to the KB (without user input).
    light_rules[color.capitalize()] = action
    print(f"Added fact: '{color.capitalize()}' ➜ '{action}'")

def block_road(a, b):
    # Add the fact Blocked(a, b) so the agent knows not to use that road.
    blocked_roads.add((a, b))
    print(f"Added fact: road {a} -> {b} is BLOCKED")

# Examples of adding facts
add_new_fact("Flashing", "Stop and check intersection")
block_road("Nakuru", "Eldoret")                                  # e.g. a road closed by flooding

# A fact that comes from the API: the current weather in Nakuru
weather = get_weather("Nakuru")
if weather:                                                      # only if the API answered
    print("Live weather in Nakuru:", weather, "->", condition_rules[weather])

def add_fact():
    # Let the user add a new traffic light colour and its action.
    new_light = input("Enter new traffic light color: ").capitalize()
    new_action = input(f"What should the agent do when light is {new_light}? ")
    light_rules[new_light] = new_action
    print(f" New fact added: '{new_light}' -> '{new_action}'")

add_fact()

# View updated KB
show_kb()

Added fact: 'Flashing' ➜ 'Stop and check intersection'
Added fact: road Nakuru -> Eldoret is BLOCKED
Live weather in Nakuru: Rain -> Slow down and keep a longer distance between the car infront.
Enter new traffic light color: No Colour
What should the agent do when light is No colour? Proceed with caution at junctions and roundabaouts
 New fact added: 'No colour' -> 'Proceed with caution at junctions and roundabaouts'

--- Current Knowledge Base ---
Light rules:
  Red        ➜ Stop, it is unsafe to move.
  Yellow     ➜ Slow down and prepare to stop.
  Green      ➜ Go: it is safe to proceed.
  Flashing   ➜ Stop and check intersection
  No colour  ➜ Proceed with caution at junctions and roundabaouts
Condition rules:
  Clear      ➜ Drive safely.
  Rain       ➜ Slow down and keep a longer distance between the car infront.
  HeavyRain  ➜ Unsafe: stop and wait
Blocked roads: [('Nakuru', 'Eldoret')]
-------------------------------
